In [1]:
# Retrieve kinematic data for an episode

from datasets import load_dataset

repo_id = "lerobot/full_folding"
hf_dataset = load_dataset(repo_id, streaming=True, split="train")

hf_episode_one_samples = []

for sample in iter(hf_dataset):
  if sample['episode_index'] == 0:
    hf_episode_one_samples.append(sample)
  else:
    break

/Users/taydakov/miniforge3/envs/lerobot/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
len(hf_episode_one_samples)

1505

In [19]:
# Visualization with OpenArm v1

import time
import numpy as np
import mujoco
import mujoco.viewer
from datasets import load_dataset
from kinematic_mappers import map_full_folding_dataset_sample_to_openarm_v1_mujoco_qpos

repo_id = "lerobot/full_folding"
mjcf_path = "../../arms/my_copy_openarm/v1/scene.xml"

# Load model and dataset
model = mujoco.MjModel.from_xml_path(mjcf_path)
data = mujoco.MjData(model)

# Create a clean lookup table matching MuJoCo joint names to their qpos memory address
mujoco_joint_adrs = {}
for i in range(model.njnt):
    name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_JOINT, i)
    mujoco_joint_adrs[name] = model.jnt_qposadr[i]

left_hand_tcp_site_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, "openarm_left_hand_tcp")
right_hand_tcp_site_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, "openarm_right_hand_tcp")
print("left_hand_tcp_site_id", left_hand_tcp_site_id)
print("right_hand_tcp_site_id", right_hand_tcp_site_id)

# Launch the corrected loop
with mujoco.viewer.launch_passive(model, data) as viewer:
    time_step = 1.0 / 30.0
    frame_index = 0

    # Initial camera position
    viewer.cam.lookat = [0.07, 0.03, 0.47]
    viewer.cam.azimuth = -0.2
    viewer.cam.elevation = -87.9
    viewer.cam.distance = 0.88

    for sample in hf_episode_one_samples:
        if not viewer.is_running():
            break
        loop_start = time.time()

        # actual_state = sample["observation.state"]

        # Reset velocities to stop any accumulated physics drifting
        data.qvel[:] = 0.0

        # Route the data explicitly through your index map
        # for ds_idx, qpos_adr in idx_map.items():
        #     data.qpos[qpos_adr] = math.radians(actual_state[ds_idx])
        data.qpos = map_full_folding_dataset_sample_to_openarm_v1_mujoco_qpos(sample)

        left_hand_tcp_pos = data.site_xpos[left_hand_tcp_site_id]
        right_hand_tcp_pos = data.site_xpos[right_hand_tcp_site_id]
        viewer.user_scn.ngeom = 0  # Clear previous frame's markers
        mujoco.mjv_initGeom(
            viewer.user_scn.geoms[0],
            type=mujoco.mjtGeom.mjGEOM_SPHERE,
            size=[0.01, 0, 0],
            pos=left_hand_tcp_pos,
            mat=np.eye(3).flatten(),
            rgba=[1, 0, 0, 1]
        )
        mujoco.mjv_initGeom(
            viewer.user_scn.geoms[1],
            type=mujoco.mjtGeom.mjGEOM_SPHERE,
            size=[0.01, 0, 0],
            pos=right_hand_tcp_pos,
            mat=np.eye(3).flatten(),
            rgba=[0, 0, 1, 1]
        )
        viewer.user_scn.ngeom = 2

        # Retrieve current camera position
        lookat = viewer.cam.lookat
        azimuth = viewer.cam.azimuth
        elevation = viewer.cam.elevation
        distance = viewer.cam.distance

        text_overlay = [
            (mujoco.mjtGridPos.mjGRID_TOPLEFT, 0, f"Current Frame: {frame_index}", ""),
            (mujoco.mjtGridPos.mjGRID_TOPLEFT, 1, f"Sim Time: {(frame_index * time_step):0.2f}s", ""),
            # (mujoco.mjtGridPos.mjGRID_TOPLEFT, 0, f"LookAt: [{lookat[0]:.2f}, {lookat[1]:.2f}, {lookat[2]:.2f}]", ""),
            # (mujoco.mjtGridPos.mjGRID_TOPLEFT, 1, f"Az/El: {azimuth:.1f} / {elevation:.1f}", ""),
            # (mujoco.mjtGridPos.mjGRID_TOPLEFT, 2, f"Dist: {distance:.2f}m", "")
        ]
        viewer.set_texts(text_overlay)

        # Run forward position kinematics
        mujoco.mj_fwdPosition(model, data)
        viewer.sync()

        frame_index += 1
        elapsed = time.time() - loop_start
        if elapsed < time_step:
            time.sleep(time_step - elapsed)

left_hand_tcp_site_id 1
right_hand_tcp_site_id 2
